In [22]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import coint
from statsmodels.regression.linear_model import OLS

# Download data
start_date = '2025-06-20'
end_date = '2025-06-24'
btc_data = yf.download('BTC-USD', start=start_date, end=end_date, interval = '1m', auto_adjust=False)
gold_data = yf.download('GC=F', start=start_date, end=end_date,interval='1m', auto_adjust=False)

# Extract Close prices as Series
btc = btc_data['Close'].squeeze()  # Ensure Series
gold = gold_data['Close'].squeeze()  # Ensure Series

# Debug: Check data types and shapes
print("BTC type:", type(btc), "BTC shape:", btc.shape)
print("Gold type:", type(gold), "Gold shape:", gold.shape)

# Check for empty data
if btc.empty or gold.empty:
    raise ValueError("One or both datasets are empty. Check yfinance data retrieval.")

# Align data into a DataFrame with proper index
data = pd.DataFrame({'BTC': btc, 'Gold': gold}).dropna()

# Debug: Check data shapes
print("Data shape:", data.shape)
print("Data columns:", data.columns)

# Check for cointegration
score, p_value, _ = coint(data['BTC'], data['Gold'])
print(f'Cointegration p-value: {p_value}')

if p_value < 0.05:
    # Fit OLS to find hedge ratio
    model = OLS(data['BTC'], data['Gold']).fit()
    hedge_ratio = model.params.iloc[0]  # Safe access to hedge ratio
    print(f'Hedge Ratio: {hedge_ratio}')

    # Ensure inputs are Series and calculate spread
    btc_series = data['BTC']  # Ensure Series
    gold_series = data['Gold']  # Ensure Series
    spread = btc_series - (hedge_ratio * gold_series)  # Explicit Series operation

    # Debug: Check spread type and shape
    print("Spread type:", type(spread))
    print("Spread shape:", spread.shape)

    # Calculate z-score
    z_score = (spread - spread.mean()) / spread.std()

    # Initialize signals DataFrame
    signals = pd.DataFrame(index=data.index)
    signals['Spread'] = spread  # Assign Series
    signals['Z_Score'] = z_score
    signals['Position'] = 0

    # Generate signals
    entry_threshold = 1
    exit_threshold = 0.5
    signals.loc[z_score > entry_threshold, 'Position'] = -1  # Short spread
    signals.loc[z_score < -entry_threshold, 'Position'] = 1   # Long spread
    signals.loc[abs(z_score) < exit_threshold, 'Position'] = 0  # Exit

    # Forward fill positions
    signals['Position'] = signals['Position'].ffill().fillna(0)

    # Calculate returns
    data['BTC_Returns'] = data['BTC'].pct_change()
    data['Gold_Returns'] = data['Gold'].pct_change()

    # Portfolio returns: long/short BTC, short/long Gold
    signals['Portfolio_Returns'] = signals['Position'].shift(1) * (
        data['BTC_Returns'] - hedge_ratio * data['Gold_Returns']
    )

    # Cumulative returns
    signals['Cumulative_Returns'] = (1 + signals['Portfolio_Returns']).cumprod()

    # Performance metrics
    total_return = signals['Cumulative_Returns'].iloc[-1] - 1
    sharpe_ratio = signals['Portfolio_Returns'].mean() / signals['Portfolio_Returns'].std() * np.sqrt(252)

    print(f'Total Return: {total_return:.2%}')
    print(f'Sharpe Ratio: {sharpe_ratio:.2f}')

    # Plot results
    plt.figure(figsize=(12, 8))
    plt.subplot(3, 1, 1)
    plt.plot(signals['Spread'], label='Spread')
    plt.title('Spread')
    plt.legend()

    plt.subplot(3, 1, 2)
    plt.plot(signals['Z_Score'], label='Z-Score')
    plt.axhline(entry_threshold, color='r', linestyle='--')
    plt.axhline(-entry_threshold, color='r', linestyle='--')
    plt.axhline(exit_threshold, color='g', linestyle='--')
    plt.axhline(-exit_threshold, color='g', linestyle='--')
    plt.title('Z-Score')
    plt.legend()

    plt.subplot(3, 1, 3)
    plt.plot(signals['Cumulative_Returns'], label='Cumulative Returns')
    plt.title('Cumulative Returns')
    plt.legend()

    plt.tight_layout()
    plt.show()
else:
    print('No cointegration found. Strategy not viable.')

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed

BTC type: <class 'pandas.core.series.Series'> BTC shape: (4512,)
Gold type: <class 'pandas.core.series.Series'> Gold shape: (2721,)
Data shape: (2002, 2)
Data columns: Index(['BTC', 'Gold'], dtype='object')
Cointegration p-value: 0.6178195328743483
No cointegration found. Strategy not viable.
